In [ ]:
import os, sys, random, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import optuna

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:

df = pd.read_csv("rfe_dataset_2019_2025.csv",
                 parse_dates=["datetime"], index_col="datetime")

y_all = df["price"].astype(float)
X_all = df.drop(columns=["price"]).astype(float)

N = len(df)
n_test = int(np.ceil(0.10 * N))
n_trainval = N - n_test

X_trainval = X_all.iloc[:n_trainval].copy()
y_trainval = y_all.iloc[:n_trainval].copy()
X_test = X_all.iloc[n_trainval:].copy()
y_test = y_all.iloc[n_trainval:].copy()

X_trainval_scaled = X_trainval.to_numpy()
y_trainval_scaled = y_trainval.to_numpy()
X_test_scaled = X_test.to_numpy()
y_test_scaled = y_test.to_numpy()


# x_scaler = MinMaxScaler()
# y_scaler = MinMaxScaler()

# X_trainval_scaled = x_scaler.fit_transform(X_trainval)
# y_trainval_scaled = y_scaler.fit_transform(y_trainval.to_numpy().reshape(-1, 1)).reshape(-1)
# X_test_scaled = x_scaler.transform(X_test)
# y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).reshape(-1)



In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X_array, y_array, seq_len):
        X_array = np.asarray(X_array, dtype=np.float32)
        y_array = np.asarray(y_array, dtype=np.float32).reshape(-1)
        self.X = X_array
        self.y = y_array
        self.seq_len = int(seq_len)
        self.length = len(self.y) - self.seq_len
        if self.length <= 0:
            raise ValueError(f"seq_len={seq_len} too long for series length={len(self.y)}")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        x_seq = self.X[idx: idx + self.seq_len]
        y_t = self.y[idx + self.seq_len]
        return torch.from_numpy(x_seq), torch.tensor(y_t, dtype=torch.float32)


def make_loader(X, y, seq_len, batch_size, shuffle=False):
    ds = SequenceDataset(X, y, seq_len)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)


def make_loader_with_tail(X_train, y_train, X_val, y_val, seq_len, batch_size):
    X_cat = np.vstack([X_train[-seq_len:], X_val])
    y_cat = np.concatenate([y_train[-seq_len:], y_val])
    ds = SequenceDataset(X_cat, y_cat, seq_len)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False)

In [ ]:
# LSTM with internal normalization
class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.2, output_size=1):
        super().__init__()

        
        self.register_buffer('x_min', torch.zeros(1, 1, n_features))
        self.register_buffer('x_max', torch.ones(1, 1, n_features))
        self.register_buffer('y_min', torch.zeros(1, 1, output_size))
        self.register_buffer('y_max', torch.ones(1, 1, output_size))
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, output_size)

    
    # Initialize normalization based on training data
    
    def init_norm(self, X_train, y_train):
        """
        Store feature-wise and target-wise min/max in model buffers.
        """
        self.x_min = X_train.amin(dim=(0, 1), keepdim=True)
        self.x_max = X_train.amax(dim=(0, 1), keepdim=True)
        self.y_min = y_train.amin(dim=0, keepdim=True)
        self.y_max = y_train.amax(dim=0, keepdim=True)

    
    # De-normalize 
    
    def target_denorm(self, y_pred):
        return y_pred * (self.y_max - self.y_min) + self.y_min

   
    def forward(self, x):
        # Normalize input inside the model
        x = (x - self.x_min) / (self.x_max - self.x_min + 1e-8)  
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1:, :])# use last time step
        out = out.squeeze(-1).squeeze(-1)
        return out


In [ ]:
def train_one_model(X_tr, y_tr, X_va, y_va, seq_len, model_params, max_epochs=60, patience=8):
    n_features = X_tr.shape[1]
    # model = LSTMRegressor(
    #     n_features=n_features,
    #     hidden_size=model_params["hidden_size"],
    #     num_layers=model_params["num_layers"],
    #     dropout=model_params["dropout"]
    # ).to(DEVICE)

    # train_loader = make_loader(X_tr, y_tr, seq_len, batch_size=model_params["batch_size"], shuffle=True)
    # val_loader   = make_loader_with_tail(X_tr, y_tr, X_va, y_va, seq_len, batch_size=model_params["batch_size"])

    # criterion = nn.MSELoss()
    # optimizer = torch.optim.Adam(model.parameters(), lr=model_params["lr"], weight_decay=1e-5)

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr.reshape(-1, 1), dtype=torch.float32)
    X_va_t = torch.tensor(X_va, dtype=torch.float32)
    y_va_t = torch.tensor(y_va.reshape(-1, 1), dtype=torch.float32)

    model = LSTMRegressor(
        n_features=n_features,
        hidden_size=model_params["hidden_size"],
        num_layers=model_params["num_layers"],
        dropout=model_params["dropout"]
    ).to(DEVICE)

    # Initialize normalization based on training data
    model.init_norm(X_tr_t, y_tr_t)

    # Normalize y
    y_tr_norm = (y_tr_t - model.y_min.squeeze(0)) / (model.y_max.squeeze(0) - model.y_min.squeeze(0) + 1e-8)
    y_va_norm = (y_va_t - model.y_min.squeeze(0)) / (model.y_max.squeeze(0) - model.y_min.squeeze(0) + 1e-8)

    train_loader = make_loader(X_tr_t.detach().numpy(), y_tr_norm.detach().numpy().flatten(), seq_len, model_params["batch_size"], shuffle=True)
    val_loader   = make_loader_with_tail(X_tr_t.numpy(), y_tr_norm.numpy().flatten(), X_va_t.numpy(), y_va_norm.numpy().flatten(), seq_len, batch_size=model_params["batch_size"])

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=model_params["lr"], weight_decay=1e-5)


    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        tr_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            y_pred = model(xb)
            loss = criterion(y_pred, yb)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * len(xb)
        tr_loss /= len(train_loader.dataset)

        model.eval()
        va_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                y_pred = model(xb)
                loss = criterion(y_pred, yb)
                va_loss += loss.item() * len(xb)
        va_loss /= len(val_loader.dataset)

        if va_loss < best_val_loss - 1e-8:
            best_val_loss = va_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val_loss

In [ ]:
# Rolling-window CV with Optuna
def objective(trial):
    hidden_size = trial.suggest_int("hidden_size", 32, 256)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])

    params = {
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "lr": lr,
        "batch_size": batch_size,
    }

    seq_len = 24*7
    n_splits = 3
    train_ratio = 0.8
    N = len(X_trainval_scaled)
    fold_size = int((1 - train_ratio) * N / n_splits)
    train_size = int(train_ratio * N)
    val_losses = []

    for i in range(n_splits):
        train_end = train_size + i * fold_size
        val_start = train_end
        val_end = val_start + fold_size
        if val_end > len(X_trainval_scaled):
            break

        X_train = X_trainval_scaled[:train_end]
        y_train = y_trainval_scaled[:train_end]
        X_val = X_trainval_scaled[val_start:val_end]
        y_val = y_trainval_scaled[val_start:val_end]

        fold_val_loss = train_one_model(X_train, y_train, X_val, y_val,
                                        seq_len=seq_len, model_params=params,
                                        max_epochs=40, patience=6)
        val_losses.append(fold_val_loss)
        # report intermediate value to enable pruning
        trial.report(fold_val_loss, i)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(val_losses))

In [ ]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(multivariate=True, seed=42), pruner=optuna.pruners.HyperbandPruner(
        min_resource=1,
        max_resource=3,
    ),
    study_name="lstm_optuna_tpe_hyperband")
study.optimize(objective, n_trials=250, timeout=None)

best_params = study.best_params
print("\nBest hyperparameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")

[I 2025-11-04 03:14:10,089] A new study created in memory with name: no-name-95337d5e-ba2c-49e3-a071-230780a864b2
[I 2025-11-04 03:21:12,313] Trial 0 finished with value: 0.001173962647271404 and parameters: {'hidden_size': 116, 'num_layers': 3, 'dropout': 0.36599697090570255, 'lr': 0.0015751320499779737, 'batch_size': 128}. Best is trial 0 with value: 0.001173962647271404.
[I 2025-11-04 04:05:45,114] Trial 1 finished with value: 0.001361483092643362 and parameters: {'hidden_size': 167, 'num_layers': 3, 'dropout': 0.010292247147901223, 'lr': 0.008706020878304856, 'batch_size': 16}. Best is trial 0 with value: 0.001173962647271404.
[I 2025-11-04 04:14:56,745] Trial 2 finished with value: 0.0010406344853646642 and parameters: {'hidden_size': 100, 'num_layers': 2, 'dropout': 0.21597250932105788, 'lr': 0.0003823475224675188, 'batch_size': 16}. Best is trial 2 with value: 0.0010406344853646642.
[I 2025-11-04 04:21:17,604] Trial 3 finished with value: 0.0010256890506225628 and parameters: {'


Best hyperparameters:
hidden_size: 59
num_layers: 2
dropout: 0.017194260557609198
lr: 0.006586289317583112
batch_size: 32


In [ ]:
split_idx = int(len(X_trainval_scaled) * 0.8)
X_tr_final, y_tr_final = X_trainval_scaled[:split_idx], y_trainval_scaled[:split_idx]
X_va_final, y_va_final = X_trainval_scaled[split_idx:], y_trainval_scaled[split_idx:]

final_model_params = best_params
final_model = LSTMRegressor(
    n_features=X_tr_final.shape[1],
    hidden_size=final_model_params["hidden_size"],
    num_layers=final_model_params["num_layers"],
    dropout=final_model_params["dropout"]
).to(DEVICE)

_ = train_one_model(X_tr_final, y_tr_final, X_va_final, y_va_final,
                    seq_len=24, model_params=final_model_params,
                    max_epochs=60, patience=8)

In [ ]:
test_loader = make_loader(X_test_scaled, y_test_scaled, seq_len=24*7, pred_len=24,
                          batch_size=final_model_params["batch_size"])

final_model.eval()
final_model.init_norm(torch.tensor(X_tr_final, dtype=torch.float32),
                      torch.tensor(y_tr_final.reshape(-1, 1), dtype=torch.float32))

preds_norm, actuals = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE).reshape(-1, 1)
        out_norm = final_model(xb)
        preds_norm.extend(out_norm.cpu().numpy())
        actuals.extend(yb.cpu().numpy())


preds_norm_t = torch.tensor(preds_norm, dtype=torch.float32)
actuals_t    = torch.tensor(actuals, dtype=torch.float32)


y_pred_inv = final_model.target_denorm(preds_norm_t).numpy().flatten()
y_true_inv = actuals_t.numpy().flatten()

rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
print(f"\nFinal Test RMSE: {rmse:.4f}")



Final Test RMSE: 72.3992


/tmp/ipykernel_3664844/1673363299.py:23: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  actuals_t    = torch.tensor(actuals, dtype=torch.float32)


In [11]:
# print(y_pred_inv)

In [12]:
# print(y_true_inv)

In [ ]:
model_dir = "./models"
os.makedirs(model_dir, exist_ok=True)  

# Save the trained LSTM model
model_path = os.path.join(model_dir, "lstm_optuna_model.pth")
torch.save(final_model.state_dict(), model_path)
print(f"Model saved successfully to: {model_path}")

# Save the fitted scalers (can inverse transform predictions later)
# x_scaler_path = os.path.join(model_dir, "x_scaler_5year.pkl")
# y_scaler_path = os.path.join(model_dir, "y_scaler_5year.pkl")

# joblib.dump(x_scaler, x_scaler_path)
# joblib.dump(y_scaler, y_scaler_path)
# print(f"Scalers saved successfully to: {x_scaler_path}, {y_scaler_path}")

Model saved successfully to: ./models/lstm_optuna_model.pth
